# Modul 4: Tool Use & Kontrollierte Autonomie

**Agent Systems: Vom Konzept zum eigenen Orchestrator**

In diesem Notebook baust du eine Tool-Registry mit Approval-Policies.

🎯 **Lernziel:** Tools nach Read-only vs. Side-Effects klassifizieren und Approval-Policies implementieren.

---

## Tool-Taxonomy: Read-only vs. Side-Effects

Nicht alle Tools sind gleich gefährlich:

```
         SICHER                    GEFÄHRLICH
  ┌─────────────────┐      ┌─────────────────────┐
  │   READ-ONLY     │      │    SIDE-EFFECTS      │
  │                 │      │                       │
  │  web_search     │      │  file_write           │
  │  file_read      │      │  email_send           │
  │  weather_check  │      │  exec (shell)         │
  │  calendar_read  │      │  database_delete      │
  └─────────────────┘      └─────────────────────┘
     → Auto-Allow            → Approval Required!
```

**MCP (Model Context Protocol)** standardisiert, wie Agents Tools entdecken und aufrufen — *"MCP ist für Agents, was USB für Peripheriegeräte ist."*

In [ ]:
# Tool-Registry mit Approval-Policies

class Tool:
    """Ein einzelnes Tool mit Metadata."""
    def __init__(self, name, description, category, function):
        self.name = name
        self.description = description
        self.category = category  # 'read-only' oder 'side-effect'
        self.function = function
    
    def __repr__(self):
        icon = '🔍' if self.category == 'read-only' else '⚠️'
        return f'{icon} {self.name} [{self.category}]'


class ToolRegistry:
    """Registry verwaltet alle verfügbaren Tools."""
    
    def __init__(self):
        self.tools = {}
    
    def register(self, tool):
        self.tools[tool.name] = tool
        print(f'  Registriert: {tool}')
    
    def get(self, name):
        return self.tools.get(name)
    
    def list_by_category(self, category):
        return [t for t in self.tools.values() if t.category == category]
    
    def show_all(self):
        print('\n📋 Tool-Registry:')
        for t in self.tools.values():
            print(f'  {t} — {t.description}')


class ApprovalPolicy:
    """Steuert welche Tools automatisch erlaubt sind."""
    
    DENY = 'deny'        # Nichts erlaubt
    ALLOWLIST = 'allowlist'  # Nur bestimmte Tools
    FULL = 'full'        # Alles erlaubt
    
    def __init__(self, mode='allowlist'):
        self.mode = mode
        self.auto_approve = set()  # Tool-Namen die auto-approved sind
    
    def allow(self, tool_name):
        self.auto_approve.add(tool_name)
    
    def check(self, tool):
        """Prüft ob ein Tool ausgeführt werden darf."""
        if self.mode == self.DENY:
            return False, 'Policy: DENY — alle Tools blockiert'
        
        if self.mode == self.FULL:
            return True, 'Policy: FULL — automatisch erlaubt'
        
        # ALLOWLIST-Modus
        if tool.name in self.auto_approve:
            return True, f'Tool "{tool.name}" ist auf der Allowlist'
        
        if tool.category == 'read-only':
            return True, f'Read-only Tool "{tool.name}" automatisch erlaubt'
        
        return False, f'⚠️ Tool "{tool.name}" braucht menschliche Freigabe!'


# === Demo: Registry aufbauen ===
print('=== Tool-Registry aufbauen ===\n')
registry = ToolRegistry()

# Read-only Tools
registry.register(Tool('web_search', 'Im Web suchen', 'read-only',
    lambda q: f'Suchergebnis für: {q}'))
registry.register(Tool('file_read', 'Datei lesen', 'read-only',
    lambda path: f'Inhalt von {path}'))
registry.register(Tool('weather', 'Wetter abfragen', 'read-only',
    lambda city: f'{city}: 18°C, sonnig'))

# Side-Effect Tools
registry.register(Tool('file_write', 'Datei schreiben', 'side-effect',
    lambda path, content: f'Geschrieben: {path}'))
registry.register(Tool('email_send', 'E-Mail senden', 'side-effect',
    lambda to, subject: f'E-Mail an {to}: {subject}'))
registry.register(Tool('exec_shell', 'Shell-Befehl ausführen', 'side-effect',
    lambda cmd: f'Ausgeführt: {cmd}'))

registry.show_all()

In [ ]:
# Approval-Policies in Aktion

def simulate_tool_call(registry, policy, tool_name, human_approves=True):
    """Simuliert einen Tool-Aufruf mit Approval-Check."""
    tool = registry.get(tool_name)
    if not tool:
        print(f'❌ Tool "{tool_name}" nicht gefunden!')
        return
    
    approved, reason = policy.check(tool)
    print(f'\n🔧 Tool-Aufruf: {tool.name}')
    print(f'   Kategorie: {tool.category}')
    print(f'   Policy-Check: {reason}')
    
    if not approved:
        # Menschliche Freigabe simulieren
        print(f'   👤 Mensch gefragt → {"✅ Freigegeben" if human_approves else "❌ Abgelehnt"}')
        approved = human_approves
    
    if approved:
        print(f'   ✅ Ausgeführt!')
    else:
        print(f'   🚫 Blockiert.')


# === Test: Drei Policy-Modi ===

print('=' * 50)
print('Policy-Modus: ALLOWLIST (Standard)')
print('=' * 50)
policy = ApprovalPolicy('allowlist')
simulate_tool_call(registry, policy, 'web_search')    # Read-only → auto
simulate_tool_call(registry, policy, 'file_write')     # Side-effect → Freigabe nötig
simulate_tool_call(registry, policy, 'email_send', human_approves=False)  # Abgelehnt!

print('\n' + '=' * 50)
print('Policy-Modus: DENY (Paranoid)')
print('=' * 50)
policy_deny = ApprovalPolicy('deny')
simulate_tool_call(registry, policy_deny, 'web_search')  # Alles blockiert

print('\n' + '=' * 50)
print('Policy-Modus: FULL (YOLO)')
print('=' * 50)
policy_full = ApprovalPolicy('full')
simulate_tool_call(registry, policy_full, 'exec_shell')  # Alles erlaubt (gefährlich!)

In [ ]:
# 🎯 ÜBUNG: Baue ein "Balanced"-Profil
#
# Aufgabe: Erstelle eine Policy die:
# 1. Alle read-only Tools automatisch erlaubt (schon implementiert)
# 2. 'file_write' auf die Allowlist setzt (auto-approved)
# 3. 'email_send' und 'exec_shell' Freigabe brauchen
# 4. Teste alle 6 Tools und zeige das Ergebnis als Tabelle

policy_balanced = ApprovalPolicy('allowlist')
# TODO: file_write zur Allowlist hinzufügen
# policy_balanced.allow(...)

# TODO: Teste alle Tools und zeige eine Übersicht
# Tipp: Iteriere über registry.tools und rufe policy_balanced.check() auf
for name, tool in registry.tools.items():
    pass  # ← Ersetze mit policy_balanced.check(tool) und print

In [ ]:
# ✅ LÖSUNG

policy_balanced = ApprovalPolicy('allowlist')
policy_balanced.allow('file_write')  # Dateien schreiben erlaubt

print('📋 Balanced-Profil: Tool-Übersicht')
print(f'{"Tool":<15} {"Kategorie":<15} {"Auto-Approved?":<20} {"Grund"}')
print('-' * 70)

for name, tool in registry.tools.items():
    approved, reason = policy_balanced.check(tool)
    status = '✅ Ja' if approved else '🔒 Nein'
    print(f'{name:<15} {tool.category:<15} {status:<20} {reason}')

print('\n💡 Fazit: Read-only + file_write sind frei, email_send und exec_shell brauchen Freigabe.')

## Die HITL-Faustregel

> *"Blinker darfst du selbst, Überholen fragen!"*

| Aktion | Risiko | Policy |
|--------|--------|--------|
| Lesen, Suchen | Niedrig | Auto-Allow |
| Dateien schreiben | Mittel | Allowlist |
| E-Mails senden | Hoch | Approval |
| Shell-Befehle | Sehr hoch | Approval + Audit |

---

### ✅ Self-Check
- [ ] Ich kann Read-only vs. Side-Effect Tools unterscheiden
- [ ] Ich verstehe die 3 Policy-Modi (Deny, Allowlist, Full)
- [ ] Ich kann erklären warum HITL bei Side-Effects wichtig ist

**→ Weiter zu [Modul 5: Persona & Workspace](modul5_persona.ipynb)**